# Occupancy Volume Representation with Neural Fields

Train neural field models to represent a 3D occupancy volume as a continuous function
mapping `(x, y, z)` coordinates to binary occupancy values.

## Setup

In [ ]:
import math
import tempfile
from pathlib import Path

import torch
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import neurofield as nf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load Data

In [ ]:
ITEM_ID = "thai_statue"
DATA_PATH = f"../data/occupancy/{ITEM_ID}.ply"
RESOLUTION = 256

## Model Configurations

Every neural field has about 133k parameters, so results compare architectures
at equal capacity. Hyperparameters follow each method's official code;
`# deviation` comments mark the few changes needed under this protocol.

In [ ]:
CONFIGS = [
    {
        "name": "RFF",  # Tancik et al. (NeurIPS 2020), 3D occupancy
        "class": nf.RFF,
        "kwargs": {
            "hidden_features": 188,
            "hidden_layers": 2,
            "num_frequencies": 256,
            "sigma": 6.0,  # sigma 12 on [0, 1] coordinates equals 6 on [-1, 1]
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,
    },
    {
        "name": "PE-MLP",  # FINER benchmark architecture
        "class": nf.PEMLP,
        "kwargs": {
            "hidden_features": 240,
            "hidden_layers": 3,
            "num_frequencies": 10,
            "output_activation": torch.tanh,
        },
        "lr": 3e-3,  # deviation: the published 1e-3 and 5e-4 lose 1-3 IoU points here
    },
    {
        "name": "MFN",  # GaborNet (Fathony et al., ICLR 2021) defaults
        "class": nf.MFN,
        "kwargs": {
            "hidden_features": 252,
            "hidden_layers": 3,
            "input_scale": 256.0,
            "alpha": 6.0,
            "beta": 1.0,
            "weight_scale": 1.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "SIREN",  # FINER benchmark
        "class": nf.SIREN,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 3,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,  # the SDF lr 1e-4 is too slow for 2000 epochs
    },
    {
        "name": "Gauss",  # FINER benchmark
        "class": nf.Gauss,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 3,
            "scale": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,  # deviation: the published 1e-4 and 5e-3 lose 5-10 IoU points
    },
    {
        "name": "WIRE",  # WIRE occupancy script, real Gabor layer
        "class": nf.RealWIRE,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 3,
            "omega": 10.0,
            "scale": 40.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,  # deviation: the official 5e-3 diverges for the real Gabor layer
    },
    {
        "name": "FINER",  # FINER benchmark
        "class": nf.FINER,
        "kwargs": {
            "hidden_features": 256,
            "hidden_layers": 3,
            "omega": 30.0,
            "output_activation": torch.tanh,
        },
        "lr": 1e-3,  # the SDF lr 1e-4 is too slow for 2000 epochs
    },
    {
        "name": "Instant-NGP",  # instant-ngp configs/volume/base.json architecture
        "class": nf.InstantNGP,
        "kwargs": {
            "num_levels": 16,
            "features_per_level": 2,
            "log2_hashmap_size": 12,
            "base_resolution": 16,
            "max_resolution": RESOLUTION,
            # The hash tables alone hold 131,072 parameters, so the MLP is
            # narrowed from 64 to 24 to match the other models.
            "hidden_features": 24,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        # deviation: the volume lr 1e-4 (tuned for long schedules) loses 9 IoU
        # points; 1e-2 is the reference lr for images and NeRF.
        "lr": 1e-2,
        "train_kwargs": {"adam_betas": (0.9, 0.99), "adam_eps": 1e-15},
    },
    {
        "name": "TensoRF-CP",
        "class": nf.TensoRF,
        "kwargs": {
            "rank": 128,
            "resolution": 256,
            "mode": "cp",
            "hidden_features": 128,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,  # 3e-2 diverges on three shapes
    },
    {
        "name": "TensoRF-VM",
        "class": nf.TensoRF,
        "kwargs": {
            # A VM plane costs resolution^2 parameters per rank, so at this
            # size resolution beats rank.
            "rank": 4,
            "resolution": 96,
            "mode": "vm",
            "hidden_features": 128,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "GA-Planes",
        "class": nf.GAPlanes,
        "kwargs": {
            "features": 25,
            "resolution": (128, 32, 8),
            "hidden_features": 128,
            "hidden_layers": 2,
            "output_activation": torch.tanh,
        },
        "lr": 3e-2,
    },
    {
        "name": "FUTON-cosine",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("cosine", {"num_components": 128, "grid_size": RESOLUTION}),
            "combiner": ("cp", {"rank": 218}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "FUTON-sinc",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("sinc", {"num_components": 128, "grid_size": RESOLUTION}),
            "combiner": ("cp", {"rank": 218}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
    {
        "name": "FUTON-lanczos",
        "class": nf.FUTON,
        "kwargs": {
            "basis": ("lanczos", {"num_components": 128, "radius": 3}),
            "combiner": ("cp", {"rank": 218}),
            "decoder": ("mlp", {"hidden_layers": 1}),
            "output_activation": torch.tanh,
        },
        "lr": 1e-2,
    },
]

## Train Models

In [ ]:
results = []

for config in CONFIGS:
    torch.manual_seed(0)

    train_dataset = nf.OccupancyCoordinateDataset(
        DATA_PATH, resolution=RESOLUTION, subsample=0.01, item_id=ITEM_ID
    )
    eval_dataset = nf.OccupancyCoordinateDataset(
        DATA_PATH, resolution=RESOLUTION, item_id=ITEM_ID
    )

    model = config["class"](in_features=3, out_features=1, **config["kwargs"])
    num_params = nf.count_parameters(model, trainable_only=False)

    print(f"\n--- {config['name']} ({num_params:,} params) ---")
    res = nf.train(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        chunk_size=512**2,
        lr=config["lr"],
        num_epochs=2000,
        metrics={"iou": nf.iou},
        log_interval=100,
        eval_interval=100,
        device=device,
        **config.get("train_kwargs", {}),
    )

    coords = eval_dataset.input
    output = nf.chunked_inference(model, coords, chunk_size=512**2, device=device)
    recon = eval_dataset.postprocess(output.cpu())

    res["config"]["model"] = config["name"]
    res["reconstructed"] = recon
    results.append(res)

    del model, recon, res["model_dict"]
    torch.cuda.empty_cache()

## Collect Results

In [ ]:
records = []
for res in results:
    for entry in res["history"]:
        if "eval" not in entry:
            continue

        records.append(
            {
                "Model": res["config"]["model"],
                "# Params (k)": res["config"]["num_params"] / 1000,
                "Iteration": entry["epoch"],
                "Time (s)": entry["elapsed"],
                "Speed (vol/s)": 1.0 / entry["eval"]["duration"],
                "IoU (%)": entry["eval"]["iou"] * 100,
            }
        )


records = pd.DataFrame(records)

## Convergence Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(ITEM_ID, fontsize=14)

sns.lineplot(
    data=records,
    x="Iteration",
    y="IoU (%)",
    hue="Model",
    style="Model",
    markers=True,
    ax=axes[0],
)
axes[0].set_ylim(94, 100)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="lower right")

sns.lineplot(
    data=records,
    x="Time (s)",
    y="IoU (%)",
    hue="Model",
    style="Model",
    markers=True,
    ax=axes[1],
)
axes[1].set_xlim(0, 30)
axes[1].set_ylim(94, 100)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
summary = (
    records.groupby("Model")
    .agg(
        {
            "# Params (k)": "first",
            "Iteration": "last",
            "Time (s)": "last",
            "Speed (vol/s)": "mean",
            "IoU (%)": "last",
        }
    )
    .sort_values("IoU (%)", ascending=False)
    .reset_index()
)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(ITEM_ID, fontsize=14)

for ax, metric in zip(axes, ["IoU (%)", "Speed (vol/s)", "# Params (k)"]):
    df_sorted = summary.sort_values(metric, ascending=True)
    ax.barh(df_sorted["Model"], df_sorted[metric])
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## Visual Comparison

In [ ]:
iou_lookup = summary.set_index("Model")["IoU (%)"]
sorted_results = sorted(
    results, key=lambda r: iou_lookup[r["config"]["model"]], reverse=True
)

with tempfile.TemporaryDirectory() as tmpdir:
    # Render original
    orig_path = Path(tmpdir) / "original"
    eval_dataset.save(eval_dataset.original, orig_path)
    orig_img = Image.open(orig_path.with_suffix(".png"))

    # Render reconstructions
    rendered = []
    for res in sorted_results:
        name = res["config"]["model"]
        fp = Path(tmpdir) / name
        eval_dataset.save(res["reconstructed"], fp)
        rendered.append((name, Image.open(fp.with_suffix(".png"))))

    items = [("Original", orig_img)] + rendered
    num_cols = math.ceil(len(items) / 2)
    fig, axes = plt.subplots(2, num_cols, figsize=(5 * num_cols, 10), squeeze=False)
    fig.suptitle(ITEM_ID, fontsize=16)

    for idx, (name, img) in enumerate(items):
        ax = axes[idx // num_cols, idx % num_cols]
        ax.imshow(img)
        title = name if name == "Original" else f"{name}\nIoU: {iou_lookup[name]:.1f}%"
        ax.set_title(title)
        ax.axis("off")

    # Hide unused axes
    for idx in range(len(items), 2 * num_cols):
        axes[idx // num_cols, idx % num_cols].axis("off")

    plt.tight_layout()
    plt.show()